## Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# Model initialization

In [ ]:
from unsloth import FastLanguageModel 
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

# We initialize The Model First Becase We Need The tokenizer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length= 2048,
    load_in_4bit=True
)

# Data preparation

In [ ]:
EOS_TOKEN = tokenizer.eos_token

dataset = load_dataset("json",data_files="tuwaiq_dataset_100.json",split="train")

prompt_template = """السؤال: {}
الإجابة: {}"""

def format_propmts(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for inst, out in zip(instructions,outputs):
        text = prompt_template.format(inst,out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_propmts,batched=True)

print(dataset[2]["text"])

# Trainer 

## Body modules
["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

## Head module
["lm_head"]


## Tail (Embeddings Only) module
["embed_tokens"]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["lm_head"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

trainer = SFTTrainer(model=model,
                     tokenizer=tokenizer,
                     train_dataset=dataset,
                     dataset_text_field = "text",
                     max_seq_length =2048,
                     args = TrainingArguments(per_device_train_batch_size = 2,
        max_steps = 60,
        learning_rate = 2e-4,
        output_dir = "tuwaiq_head_model",
        optim = "adamw_8bit",)
                     )

trainer.train()

# Inference

In [ ]:
FastLanguageModel.for_inference(model) 

test_prompt = prompt_template.format(
    "كيف يمكنني التسجيل في أكاديمية طويق؟", 
    "" 
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("\n--- إجابة النموذج ---")
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

# Saving the Model

In [ ]:


model.save_pretrained("tuwaiq_head_model") 
tokenizer.save_pretrained("tuwaiq_head_model")

print("✅ تم حفظ النموذج بنجاح!")